# 20 · Disease-Specific View Selection × Selective EHR Residual Fusion

Hiện thực `PLAN_VIEW_SELECTION_EHR_FACTORIAL_V1.md`.

## ⚠ Ba mục chặn — đọc trước khi chạy

### 1. Chỉ có **một** image seed

§2.1 và §10 yêu cầu ba seed `{42, 1337, 2024}` và cổng *"same direction in ≥2/3 image seeds"*.
Hiện chỉ `seed 42` có checkpoint — `1337`/`2024` đã **hoãn** (amendment
`REVIEW_NB16_PREGPU.md` §5-ter), mỗi seed `12.5 h` GPU.

Notebook lặp trên **các seed có sẵn** và **assert `≥ 2` seed** tại `C8` (TEST). Với một seed,
Stage 1 chạy được đầy đủ nhưng **không** kết luận được theo cổng §10.

### 2. TEST chưa từng được chạm

Mọi plan trước đều khoá `NO TEST ACCESS`. §7.1 giờ cho phép dùng TEST **một lần**. Đó là dùng
đúng mục đích của tập test — nhưng **không đảo ngược được**. `C8` khoá sau `RUN_TEST = False`
cộng một assert rằng toàn bộ Stage 1 đã lock.

### 3. Notebook số 19 đã bị chiếm

§14 đặt tên `19_view_selection_ehr_factorial.ipynb`, nhưng `19_fov_ablation.ipynb` đã tồn tại.
Dùng **`20_`**. Thư mục artifact giữ đúng `DATA_DIR/view_ehr_factorial/`.

---

## ⚠ Hai điều đã biết trước từ nb17/nb18 — ảnh hưởng kỳ vọng

**Nhánh view rất có thể null.** nb17 `V1`/`V3` đã chạy một **siêu tập** của `A1`: `V3` là pooler
học được với 8 đặc trưng/nhãn, cross-fit 5-fold theo subject. Kết quả `macro +0.00047`,
CI chứa 0; `V1` (lưới `W_F` theo nhãn) `+0.00017`, CI chứa 0.

`A1` ở đây **hẹp hơn** `V3` (một `α_c` thay vì 8 hệ số), nên khó vượt hơn.

Cái plan này **thêm** so với nb17 là **attribution**: `AG` (global vs disease-specific), `AV`
(availability vs content), `AS` (shuffled lateral). nb17 có null nhưng không quy được nguyên
nhân. Đó mới là giá trị của nhánh view — không phải kỳ vọng lật kết quả.

**Nhánh EHR là phần có giá trị nhất.** nb18 cho `S_both − S_img`: `macro +0.00251`
CI `[+0.00151, +0.00349]`, `rare3 +0.00269` CI `[+0.00009, +0.00539]` — vượt cổng, CI loại 0.
Nhưng để lại đúng một nghi vấn: `Fracture +0.0118` có thể chỉ là **bối cảnh khám**
(`ED → chấn thương`), và `E3b` cho thấy gain tập trung ở ED (`+0.00528`).

`B1_CTX` vs `B1` và `B1_SHUF` của plan này **đúng là hai phép thử tách được nghi vấn đó**.

## ⚠ Một lệch giữa mục tiêu và hàm mục tiêu

§8 tối ưu **BCE**; §9.1 chấm bằng **Macro mAP**. Hai thứ không đồng hướng, nhất là ở nhãn hiếm.
nb17 `V1` tối ưu **thẳng AP** bằng lưới. Nên `A1` (BCE) có thể **kém hơn** `V1` (AP) trên chính
endpoint. Notebook làm theo plan (BCE, khả vi, có regularization) và **báo cáo cả hai** để đọc
được chênh lệch đó.

---

## Staging (§15)

```
Stage 0  C1-C3   cache p_F/p_L, train Q_CTX/Q_FULL, hash        CPU + 1 luot GPU (co the tai dung nb17)
Stage 1  C4-C7   OOF-VAL: AG/A1/AV/AS, B0/B1/B1_CTX/B1_SHUF     CPU, phut
Stage 2  C8      TEST mot lan, sau khi khoa                     GPU (per-image TEST) + CPU
```

## C0 · Config + prereg **khoá trước**

Mọi lưới `λ`, seed hoán vị, ràng buộc, và cổng go/no-go ghi ra **trước** khi chạy dòng nào.

In [1]:
import os, io, json, math, time, pickle, hashlib, itertools, shutil, tarfile
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')
NB16      = os.path.join(DATA_DIR, 'label_reliability')
NB17_PRED = os.path.join(DATA_DIR, 'baseline_opt', 'preds')
SC_DIR    = os.path.join(DATA_DIR, 'view_ehr_factorial')            # §14
for _d in ('manifests', 'predictions', 'models', 'metrics'):
    os.makedirs(f'{SC_DIR}/{_d}', exist_ok=True)

# ---------------- KHOA TRUOC ----------------
IMAGE_SEEDS   = [42, 1337, 2024]        # §2.1 — chi seed co checkpoint moi chay
REF_EPOCH     = 2                       # R0 ep2 = checkpoint §9 cua nb16 chon; nb17 khong tim duoc gi hon
W_F0, W_L0    = 0.7, 0.3                # §3.1
ALPHA_INIT    = 0.70                    # §3.2/§3.3 -> sigmoid(a) = 0.40
ALPHA_LO      = 0.50                    # §3.3 primary constraint
LAMBDA_G      = [0.0, 0.01, 0.1, 1.0]   # §3.4
LAMBDA_D      = [0.01, 0.1, 1.0, 10.0]
LAMBDA_B      = [0.01, 0.1, 1.0, 10.0]  # §5.3
B_INIT        = -8.0                    # §5.3 -> beta ~ 3.35e-4
PERM_SEEDS    = [101, 202, 303, 404, 505]   # §4.2 / §6.3 — 5 seed hoan vi khoa truoc
K_OUTER, K_INNER = 5, 4                 # §7.2
N_BOOT        = 2000                    # §11.1
MAX_ITER      = 500                     # §8
RARE3 = ['Pleural_Other', 'Fracture', 'Lung_Lesion']
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']
MODEL_SIZE = 518

GATES = dict(                                                        # §10
    view_primary='A1 - AG Macro >= +0.001',
    view_practical='A1 - A0 Macro (bao cao, khong phai cong)',
    view_safeguard='NonTarget-10 >= -0.0015; cung chieu >= 2/3 seed; bootstrap khong hai',
    view_rare3_optional='Rare-3 >= +0.002',
    view_content='A1 > AS  (neu A1 ~ AV thi goi la acquisition-policy/context, KHONG phai lateral content)',
    ehr_primary='B1 - A1: Macro >= +0.002 HOAC Rare-3 >= +0.002',
    ehr_safeguard='NonTarget-10 >= -0.0015; cung chieu >= 2/3 seed; B1 > B1_SHUF; bootstrap khong hai',
    combined='B1 - A0 Macro >= +0.002',
    interaction='KHONG co nguong. Bao cao diem + CI + dau qua seed')
FORBIDDEN = [                                                        # §13
    'doi label subset sau khi xem TEST', 'hard-code alpha_Fracture = 1',
    'tat EHR rieng cho Pleural_Other', 'doi rang buoc beta', 'doi lambda',
    'chon arm theo per-label TEST', 'them patient-specific gate de cuu null']

PREREG = dict(
    notebook='20_view_selection_ehr_factorial.ipynb',
    plan='PLAN_VIEW_SELECTION_EHR_FACTORIAL_V1.md',
    deviation_notebook_number='plan ghi 19_, nhung 19_fov_ablation.ipynb da ton tai -> dung 20_',
    factorial='A0/A1 x no-EHR/Q_FULL; controls AG, AV, AS, B1_CTX, B1_SHUF',
    image_seeds=IMAGE_SEEDS, ref_epoch=REF_EPOCH,
    alpha=dict(init=ALPHA_INIT, lo=ALPHA_LO, hi=1.0, map='alpha = 0.5 + 0.5*sigmoid(a)'),
    beta=dict(init_logit=B_INIT, lo=0.0, hi=1.0, map='beta = sigmoid(b)', no_intercept=True),
    lambdas=dict(g=LAMBDA_G, d=LAMBDA_D, beta=LAMBDA_B),
    perm_seeds=PERM_SEEDS, k_outer=K_OUTER, k_inner=K_INNER, n_boot=N_BOOT,
    fitting='tuan tu: fit+khoa alpha -> freeze -> fit beta (§5.4). KHONG joint trong primary.',
    objective='BCE + regularization (§8); endpoint la Macro mAP (§9.1) — HAI THU KHAC NHAU',
    lambda_selection='mAP macro tren inner-held-out; KHONG dung outer-held-out (§7.2)',
    gates=GATES, forbidden_after_test=FORBIDDEN,
    prior_evidence=dict(
        nb17_V1='W_F theo nhan, cross-fit: macro +0.00017, CI chua 0',
        nb17_V3='pooler hoc duoc (sieu tap cua A1): macro +0.00047, CI chua 0',
        nb18_E2='S_both - S_img: macro +0.00251 CI [+0.00151,+0.00349]; rare3 +0.00269 CI [+0.00009,+0.00539]',
        nb18_caveat='Fracture +0.0118 co the la boi canh ED; B1_CTX/B1_SHUF la phep thu tach dieu do'),
    test_policy='TEST chi chay MOT LAN o C8, sau khi Stage 1 khoa. RUN_TEST mac dinh False.')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json', 'w'), indent=1)

try:
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:
    dev = 'cpu'
print(f'device {dev}')
print('=== COÅNG (§10) ===')
for k, v in GATES.items(): print(f'  {k:<20} {v}')
print()
print('=== CAM SAU KHI XEM TEST (§13) ===')
for f in FORBIDDEN: print(f'  - {f}')
print()
print(f'artifacts -> {SC_DIR}')

Mounted at /content/drive
device cuda
=== COÅNG (§10) ===
  view_primary         A1 - AG Macro >= +0.001
  view_practical       A1 - A0 Macro (bao cao, khong phai cong)
  view_safeguard       NonTarget-10 >= -0.0015; cung chieu >= 2/3 seed; bootstrap khong hai
  view_rare3_optional  Rare-3 >= +0.002
  view_content         A1 > AS  (neu A1 ~ AV thi goi la acquisition-policy/context, KHONG phai lateral content)
  ehr_primary          B1 - A1: Macro >= +0.002 HOAC Rare-3 >= +0.002
  ehr_safeguard        NonTarget-10 >= -0.0015; cung chieu >= 2/3 seed; B1 > B1_SHUF; bootstrap khong hai
  combined             B1 - A0 Macro >= +0.002
  interaction          KHONG co nguong. Bao cao diem + CI + dau qua seed

=== CAM SAU KHI XEM TEST (§13) ===
  - doi label subset sau khi xem TEST
  - hard-code alpha_Fracture = 1
  - tat EHR rieng cho Pleural_Other
  - doi rang buoc beta
  - doi lambda
  - chon arm theo per-label TEST
  - them patient-specific gate de cuu null

artifacts -> /content/drive/MyDri

## C1 · Cohort · study-view manifest · fold theo subject

`p_F` / `p_L` / `m_F` / `m_L` theo §2.2. Không trộn ảnh khác study.

Fold: **5 outer × 4 inner**, group = `subject_id`, **cùng một phân hoạch cho mọi arm** (§7.2) —
nếu không thì mọi so sánh giữa arm sẽ lẫn nhiễu phân hoạch.

Manifest và fold ghi ra `manifests/` kèm hash, để `C8` assert TEST chưa từng xuất hiện trong fit.

In [2]:
meta = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; C = len(LABELS)
cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], 'float64'); N_TRAIN = int(cf['num_train'])
TIER = np.where(N_POS_TRAIN/N_TRAIN < 0.02, 'tail',
                np.where(N_POS_TRAIN/N_TRAIN < 0.10, 'medium', 'head'))
RARE3_IDX  = np.array([LABELS.index(l) for l in RARE3])
NONTGT_IDX = np.setdiff1d(np.arange(C), RARE3_IDX)
TAIL_IDX   = np.where(TIER == 'tail')[0]
W3_IDX     = np.array([LABELS.index(l) for l in WORST3_FIXED])
GRP = dict(macro=np.arange(C), rare3=RARE3_IDX, nontgt=NONTGT_IDX,
           tail=TAIL_IDX, worst3=W3_IDX)

ehr_all = (pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'),
                           columns=['study_id', 'subject_id', 'split', 'setting'] + LABELS)
           .sort_values('study_id').reset_index(drop=True))
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal', 'lateral'])]
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
MAN = (vm.drop(columns=_dup).merge(ehr_all[['study_id', 'subject_id', 'split']],
                                   on='study_id', how='inner')
       .sort_values('dicom_id').reset_index(drop=True))
print(f'cohort {len(ehr_all):,} study | anh {len(MAN):,}')
print(f'  split study: {dict(ehr_all.split.value_counts())}')
print(f'  split anh  : {dict(MAN.split.value_counts())}')

def build_split(sp):
    """Tra ve dict mo ta mot split: study id, nhan, subject, setting, chi so anh."""
    e = ehr_all[ehr_all.split == sp].reset_index(drop=True)
    sid = np.sort(e.study_id.to_numpy())
    e = e.set_index('study_id').loc[sid].reset_index()
    img = np.where(MAN.split.to_numpy() == sp)[0]
    sub_man = MAN.iloc[img]
    pos = pd.Series(sub_man.study_id.to_numpy()).map({s: i for i, s in enumerate(sid)}).to_numpy()
    return dict(split=sp, sid=sid, y=e[LABELS].to_numpy('float64'),
                subject=e.subject_id.to_numpy(), setting=e.setting.astype(str).to_numpy(),
                img_idx=img, img_study=pos,
                is_F=(sub_man.view_group.to_numpy() == 'frontal'),
                is_L=(sub_man.view_group.to_numpy() == 'lateral'))
SPL = {sp: build_split(sp) for sp in ('val', 'test')}
for sp, d in SPL.items():
    nF = np.bincount(d['img_study'][d['is_F']], minlength=len(d['sid'])) > 0
    nL = np.bincount(d['img_study'][d['is_L']], minlength=len(d['sid'])) > 0
    d['mF'], d['mL'] = nF.astype('float64'), nL.astype('float64')
    print(f'  {sp.upper():<5} {len(d["sid"]):,} study | {len(np.unique(d["subject"])):,} subject | '
          f'{len(d["img_idx"]):,} anh | F-only {int((nF&~nL).sum()):,} '
          f'F+L {int((nF&nL).sum()):,} L-only {int((~nF&nL).sum()):,}')
print(f'  -> chi study F+L moi cap gradient cho alpha (§2.2): '
      f'{int((SPL["val"]["mF"]*SPL["val"]["mL"]).sum()):,} / {len(SPL["val"]["sid"]):,} o VAL')

# ---------------- fold theo subject, CHUNG cho moi arm (§7.2) ----------------
def subject_folds(subj, K, seed):
    su = np.unique(subj); r = np.random.default_rng(seed)
    f = np.empty(len(su), 'int64'); f[r.permutation(len(su))] = np.arange(len(su)) % K
    return pd.Series(subj).map(dict(zip(su, f))).to_numpy()
V = SPL['val']
V['outer'] = subject_folds(V['subject'], K_OUTER, 20240101)
V['inner'] = np.full(len(V['sid']), -1, 'int64')
for k in range(K_OUTER):
    m = V['outer'] != k
    V['inner'][m] = subject_folds(V['subject'][m], K_INNER, 20240200+k)
for k in range(K_OUTER):
    assert set(V['subject'][V['outer'] == k]).isdisjoint(set(V['subject'][V['outer'] != k]))
for k in range(K_OUTER):
    m = np.where(V['outer'] != k)[0]
    for j in range(K_INNER):
        a = set(V['subject'][m][V['inner'][m] == j]); b = set(V['subject'][m][V['inner'][m] != j])
        assert a.isdisjoint(b), f'inner fold {j} trong outer {k} chong lap subject'
print()
print(f'fold: outer {K_OUTER} x inner {K_INNER}, group=subject_id, KHONG chong lap  OK')
print(f'  study/outer-fold: {[int((V["outer"]==k).sum()) for k in range(K_OUTER)]}')

pd.DataFrame(dict(study_id=V['sid'], subject_id=V['subject'], setting=V['setting'],
                  outer=V['outer'], inner=V['inner'], mF=V['mF'], mL=V['mL'])
             ).to_parquet(f'{SC_DIR}/manifests/val_subject_folds.parquet', index=False)
_sha = lambda o: hashlib.sha256(json.dumps(o, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
HASH = dict(val_sid=_sha(V['sid'].tolist()), test_sid=_sha(SPL['test']['sid'].tolist()),
            val_folds=_sha(V['outer'].tolist()), labels=_sha(LABELS))
json.dump(HASH, open(f'{SC_DIR}/manifests/test_manifest_hash.json', 'w'), indent=1)
print(f'  hash TEST manifest: {HASH["test_sid"][:16]}...  (chua dung o bat ky buoc fit nao)')

cohort 211,130 study | anh 346,578
  split study: {'train': np.int64(147791), 'val': np.int64(31670), 'test': np.int64(31669)}
  split anh  : {'train': np.int64(242497), 'test': np.int64(52201), 'val': np.int64(51880)}
  VAL   31,670 study | 8,956 subject | 51,880 anh | F-only 16,314 F+L 15,299 L-only 57
  TEST  31,669 study | 9,160 subject | 52,201 anh | F-only 16,333 F+L 15,275 L-only 61
  -> chi study F+L moi cap gradient cho alpha (§2.2): 15,299 / 31,670 o VAL

fold: outer 5 x inner 4, group=subject_id, KHONG chong lap  OK
  study/outer-fold: [6188, 6404, 6321, 6098, 6659]
  hash TEST manifest: 0f1abff04c75a2cb...  (chua dung o bat ky buoc fit nao)


## C2 · `p_F` / `p_L` cho từng image seed *(Stage 0, phần GPU)*

**Tái dùng cache của nb17 cho `seed 42 / VAL`** — `perimg_val_R0_ep2_center518.npy` đã có, đúng
transform `center518` và đúng thứ tự ảnh. Không cần forward lại.

Seed khác hoặc split TEST thì phải forward. `RUN_IMAGE_FORWARD` mặc định `False` để `C2` chạy
được ngay bằng cache.

### Cổng bắt buộc

`A0` dựng từ `p_F`/`p_L` của **chính notebook này** phải khớp công thức §3.1 tới `1e-8`
(assertion §8). Đây là kiểm **công thức**, không phải kiểm khớp với nb16 — nb17 đã cho thấy
so xác suất qua GPU khác sẽ lệch `~1e-3` và **không** dùng làm cổng được.

Cổng khớp nb16 dùng **Spearman** trên study-level (bền với nhiễu số học).

In [3]:
RUN_IMAGE_FORWARD = True      # True de forward seed/split chua co cache
# snapshot cua nb16: {label_reliability}/snapshots/{arm}_s{seed}_ep{ep}.pt
CKPT_R0 = {s: f'{NB16}/snapshots/R0_s{s}_ep{REF_EPOCH}.pt' for s in IMAGE_SEEDS}

def _pooled_FL(p_img, d):
    """p_img [n_anh, C] theo thu tu d['img_idx'] -> p_F, p_L [n_study, C]."""
    n = len(d['sid']); out = []
    for mask in (d['is_F'], d['is_L']):
        cnt = np.bincount(d['img_study'][mask], minlength=n).astype('float64')
        a = np.zeros((n, C))
        for j in range(C):
            a[:, j] = np.bincount(d['img_study'][mask], weights=p_img[mask, j].astype('float64'),
                                  minlength=n)/np.maximum(cnt, 1)
        out.append(a)
    return out

PFL = {}          # (seed, split) -> dict(pF, pL)
_npz = lambda s, sp: f'{SC_DIR}/predictions/image_view_probs_seed{s}_{sp}.npz'
for s in IMAGE_SEEDS:
    for sp in ('val', 'test'):
        p = _npz(s, sp)
        if os.path.exists(p):
            z = np.load(p); PFL[(s, sp)] = dict(pF=z['pF'], pL=z['pL']); continue
        if s == 42 and sp == 'val':
            _c = f'{NB17_PRED}/perimg_val_R0_ep2_center518.npy'
            if os.path.exists(_c):
                pi = np.load(_c)
                assert len(pi) == len(SPL[sp]['img_idx']), \
                    f'cache nb17 co {len(pi)} anh, manifest co {len(SPL[sp]["img_idx"])}'
                pF, pL = _pooled_FL(pi, SPL[sp])
                np.savez_compressed(p, pF=pF, pL=pL); PFL[(s, sp)] = dict(pF=pF, pL=pL)
                print(f'seed {s} {sp}: tai dung cache per-image cua nb17'); continue
        _ck = CKPT_R0.get(s)
        if _ck and os.path.exists(_ck):
            print(f'seed {s} {sp}: CHUA CO -> RUN_IMAGE_FORWARD = True la DU (~6 phut GPU)')
        else:
            print(f'seed {s} {sp}: CHUA CO -> KHONG co checkpoint R0_s{s}_ep{REF_EPOCH}.pt. '
                  f'Bat co KHONG du: phai train ~12.5 h GPU o nb16 truoc.')
HAVE = sorted({s for (s, sp) in PFL if sp == 'val'})
print()
print(f'seed co p_F/p_L tren VAL: {HAVE}')
_miss = [s for s in IMAGE_SEEDS if s not in HAVE]
if _miss:
    _nock = [s for s in _miss if not (CKPT_R0.get(s) and os.path.exists(CKPT_R0[s]))]
    print(f'  ! thieu seed {_miss} -> Stage 1 van chay duoc, nhung cong "2/3 seed" (§10)')
    print(f'    KHONG danh gia duoc.')
    if _nock:
        print(f'    Trong do {_nock} CHUA CO CHECKPOINT: nb16 chi train seed 42.')
        print(f'    -> forward khong tao ra duoc; can ~12.5 h GPU/seed de train truoc.')
assert HAVE, 'khong co seed nao -> khong the tiep tuc'

# ---------------- COÅNG: cong thuc A0 (§8) ----------------
def pool_alpha(pF, pL, mF, mL, alpha):
    """§3.1/§3.2/§3.3 — renormalize bat buoc. alpha: scalar hoac [C]."""
    a = np.broadcast_to(np.asarray(alpha, 'float64').reshape(1, -1), pF.shape)
    num = mF[:, None]*a*pF + mL[:, None]*(1-a)*pL
    den = mF[:, None]*a + mL[:, None]*(1-a)
    return num/np.maximum(den, 1e-300)
d = SPL['val']; z = PFL[(HAVE[0], 'val')]
A0_ref = (W_F0*z['pF']*d['mF'][:, None] + W_L0*z['pL']*d['mL'][:, None]) \
         / (W_F0*d['mF'] + W_L0*d['mL'])[:, None]
A0_new = pool_alpha(z['pF'], z['pL'], d['mF'], d['mL'], np.full(C, W_F0))
_e = float(np.abs(A0_ref-A0_new).max())
print()
print(f'COÅNG §8 · A0 reproduction: max_abs_diff {_e:.2e}  {"OK" if _e <= 1e-8 else "FAIL"}')
assert _e <= 1e-8
_one = (d['mF'] > 0) & (d['mL'] == 0)
_p1 = pool_alpha(z['pF'], z['pL'], d['mF'], d['mL'], np.full(C, 0.55))
_p2 = pool_alpha(z['pF'], z['pL'], d['mF'], d['mL'], np.full(C, 0.95))
_e2 = float(np.abs(_p1[_one]-_p2[_one]).max())
print(f'COÅNG §8 · study MOT view khong phu thuoc alpha: max_abs_diff {_e2:.2e} '
      f'{"OK" if _e2 <= 1e-12 else "FAIL"}')
assert _e2 <= 1e-12

# ---------------- doi chieu nb16 bang THU HANG (khong dung lech tuyet doi) ----------------
_p16 = f'{NB16}/preds/R0_s42_ep{REF_EPOCH}.npy'
if 42 in HAVE and os.path.exists(_p16):
    P16 = np.load(_p16)
    def _sp(u, v):
        ru = pd.Series(u).rank().to_numpy(); rv = pd.Series(v).rank().to_numpy()
        ru -= ru.mean(); rv -= rv.mean()
        return float((ru*rv).sum()/max(np.sqrt((ru**2).sum()*(rv**2).sum()), 1e-30))
    _rho = min(_sp(A0_new[:, j], P16[:, j]) for j in range(C))
    print(f'COÅNG · A0 vs nb16 R0_ep{REF_EPOCH}: Spearman min {_rho:.6f} '
          f'{"OK" if _rho >= 0.999 else "FAIL"}')
    print('  (dung THU HANG, khong dung lech tuyet doi — bai hoc nb17: GPU khac cho ~1e-3)')
    assert _rho >= 0.999

seed 42 test: CHUA CO -> RUN_IMAGE_FORWARD = True la DU (~6 phut GPU)
seed 1337 val: CHUA CO -> KHONG co checkpoint R0_s1337_ep2.pt. Bat co KHONG du: phai train ~12.5 h GPU o nb16 truoc.
seed 1337 test: CHUA CO -> KHONG co checkpoint R0_s1337_ep2.pt. Bat co KHONG du: phai train ~12.5 h GPU o nb16 truoc.
seed 2024 val: CHUA CO -> KHONG co checkpoint R0_s2024_ep2.pt. Bat co KHONG du: phai train ~12.5 h GPU o nb16 truoc.
seed 2024 test: CHUA CO -> KHONG co checkpoint R0_s2024_ep2.pt. Bat co KHONG du: phai train ~12.5 h GPU o nb16 truoc.

seed co p_F/p_L tren VAL: [42]
  ! thieu seed [1337, 2024] -> Stage 1 van chay duoc, nhung cong "2/3 seed" (§10)
    KHONG danh gia duoc.
    Trong do [1337, 2024] CHUA CO CHECKPOINT: nb16 chi train seed 42.
    -> forward khong tao ra duoc; can ~12.5 h GPU/seed de train truoc.

COÅNG §8 · A0 reproduction: max_abs_diff 1.11e-16  OK
COÅNG §8 · study MOT view khong phu thuoc alpha: max_abs_diff 1.11e-16 OK
COÅNG · A0 vs nb16 R0_ep2: Spearman min 1.000000 OK

## C3 · `Q_CTX` / `Q_FULL` — hai EHR-only learner *(Stage 0, CPU)*

§2.3 chia đặc trưng làm hai tầng. Đây là chỗ **tách được nghi vấn của nb18**:

| | gồm | trả lời |
|---|---|---|
| `Q_CTX` | care setting · **missingness** · đếm/độ mới của phép đo · demographics (nếu có) | *"EHR có phải chỉ là bối cảnh khám không?"* |
| `Q_FULL` | `Q_CTX` **+ giá trị** lab · vital · comorbidity · microbiology | *"giá trị lâm sàng có thêm gì không?"* |

Ranh giới đúng là **"có đo hay không"** vs **"đo ra bao nhiêu"**. `mask` thuộc `CTX`, `value`
thuộc riêng `FULL`.

nb18 `E3b` cho thấy gain Rare-3 tập trung ở `ED (+0.00528)` và âm ở `outpatient (−0.00239)` —
đúng dấu hiệu của tín hiệu bối cảnh. `B1_CTX` vs `B1` ở `C6` sẽ định lượng nó.

Cả hai learner **chỉ train trên TRAIN** (§7.3) — không đụng VAL/TEST.

**Chi phí:** 2 model × 13 nhãn XGBoost trên `147,791 × ~170` ⇒ **~8–15 phút CPU**. Cache ra
`.npz`.

In [4]:
ehr_f = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
MISSING = vocab['reserved']['MISSING']
tr_m = (ehr_f.split == 'train').to_numpy()
CTX, FULL, NM_C, NM_F = [], [], [], []
def add(block, names, tier):
    b = np.asarray(block, 'float32'); b = b[:, None] if b.ndim == 1 else b
    assert b.shape == (len(ehr_f), len(names)), (b.shape, len(names))
    (CTX if tier == 'ctx' else FULL).append(b)
    (NM_C if tier == 'ctx' else NM_F).extend(names)

# ---------------- CTX: boi canh + missingness + dem/recency (+ demographics neu co) ----------------
DEMO = [c for c in ('anchor_age', 'age', 'gender', 'sex') if c in ehr_f.columns]
if DEMO:
    _d = np.stack([pd.to_numeric(ehr_f[c], errors='coerce').fillna(-1).to_numpy('float32')
                   if ehr_f[c].dtype.kind in 'if'
                   else pd.factorize(ehr_f[c].astype(str))[0].astype('float32') for c in DEMO], 1)
    add(_d, [f'demo_{c}' for c in DEMO], 'ctx')
print(f'demographics tim thay: {DEMO if DEMO else "KHONG CO trong cohort"}')
if not DEMO:
    print('  ! §2.3 liet ke demographics cho Q_CTX nhung cohort khong co cot nao.')
    print('    Q_CTX chay khong co demographics — ghi vao bao cao nhu mot sai lech so voi plan.')

LABS = vocab['labs']
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr_f.columns]
COMO = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr_f.columns]
add(np.stack([ehr_f[f'{c}_mask'].to_numpy('float32') for c in LABS], 1),
    [f'm_lab_{c}' for c in LABS], 'ctx')
if VITALS:
    add(np.stack([ehr_f[f'{c}_mask'].to_numpy('float32') for c in VITALS], 1),
        [f'm_vit_{c}' for c in VITALS], 'ctx')
def zs(x):
    z = np.log1p(np.clip(x.astype('float32'), 0, None))
    mu, sd = z[tr_m].mean(), z[tr_m].std() or 1.0
    return ((z-mu)/sd).astype('float32')
# dem SO LAN DO / muc do su dung y te  -> CTX.  Dem KET QUA DUONG TINH -> FULL.
COUNTS = ['t2_n_pending', 't3_n_specimens',
          't4_years_since_first_culture', 't4_n_prior_encounters']
add(np.stack([zs(ehr_f[c].to_numpy()) for c in COUNTS], 1), [f'cnt_{c}' for c in COUNTS], 'ctx')
BIN = ['t2_pending', 't2_time_imprecise', 't1_same_encounter']
add(np.stack([ehr_f[b].to_numpy('float32') for b in BIN], 1), [f'bin_{b}' for b in BIN], 'ctx')
def codes(s, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return s.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('float32')
add(codes(ehr_f['setting'], ['inpatient', 'ED', 'outpatient']), ['ctx_setting'], 'ctx')
# mask "da tung DO" -> CTX.  mask "da tung DUONG TINH" -> FULL (day la ket qua).
add(ehr_f['t1_days_since_last_specimen_mask'].to_numpy('float32'),
    ['m_rec_t1_days_since_last_specimen'], 'ctx')

# ---------------- RESULT-DERIVED: BAT BUOC o FULL, KHONG duoc vao CTX ----------------
# 02_ehr_features.ipynb dinh nghia:
#   t1_status              = np.select([t1_any_positive, t1_any_cancelled, t1_any_no_growth],[1,2,3])
#   t4_n_prior_positive_ever = COUNTIF(status='POSITIVE')
#   t1/t3_days_since_last_positive = MIN(IF(status='POSITIVE', days_before, NULL))
#   t3_recency_bin         = pd.cut(t3_days_since_last_positive, ...)
# Tat ca deu mang KET QUA cay, khong phai boi canh cham soc. §6 dung B1_CTX de tach
# shortcut khoi tin hieu lam sang — de chung o CTX se thoi phong phan "shortcut".
REC_POS = ['t1_days_since_last_positive', 't3_days_since_last_positive']
add(np.stack([zs(ehr_f['t4_n_prior_positive_ever'].to_numpy())], 1),
    ['res_n_prior_positive_ever'], 'full')
add(np.stack([codes(ehr_f['t3_recency_bin'], ['none', '7-30d', '30-90d']),
              ehr_f['t1_status'].to_numpy('float32')], 1),
    ['res_recency_bin', 'res_t1_status'], 'full')
add(np.stack([ehr_f[f'{c}_mask'].to_numpy('float32') for c in REC_POS], 1),
    [f'res_m_{c}' for c in REC_POS], 'full')
REC = ['t1_days_since_last_specimen'] + REC_POS

# ---------------- FULL rieng: GIA TRI ----------------
add(np.stack([ehr_f[f'{c}_value'].to_numpy('float32') for c in LABS], 1),
    [f'v_lab_{c}' for c in LABS], 'full')
if VITALS:
    add(np.stack([ehr_f[f'{c}_value'].to_numpy('float32') for c in VITALS], 1),
        [f'v_vit_{c}' for c in VITALS], 'full')
if COMO:
    add(np.stack([ehr_f[c].to_numpy('float32') for c in COMO], 1), [f'v_como_{c}' for c in COMO], 'full')
add(np.stack([ehr_f[f'{c}_value'].to_numpy('float32') for c in REC], 1),
    [f'v_rec_{c}' for c in REC], 'full')
from collections import Counter
for nm, col in (('t1_org', 't1_org_ids'), ('t1_spec', 't1_spec_ids'),
                ('t2_spec', 't2_spec_ids'), ('t3_org', 't3_org_ids')):
    lst = [np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
           for v in ehr_f[col].to_numpy()]
    lst = [a[a != MISSING] for a in lst]
    cnt = Counter()
    for i in np.where(tr_m)[0]: cnt.update(set(lst[i].tolist()))
    top = [i for i, _ in cnt.most_common(30)]; pos = {v: k for k, v in enumerate(top)}
    M = np.zeros((len(ehr_f), len(top)+1), 'float32')
    for i, a in enumerate(lst):
        for v in set(a.tolist()):
            if v in pos: M[i, pos[v]] = 1.0
        M[i, -1] = len(a)
    add(M, [f'v_{nm}_id{v}' for v in top]+[f'v_{nm}_n'], 'full')

X_CTX = np.nan_to_num(np.concatenate(CTX, 1), nan=0., posinf=0., neginf=0.)
X_FULL = np.nan_to_num(np.concatenate([X_CTX]+FULL, 1), nan=0., posinf=0., neginf=0.)
Y_ALL = ehr_f[LABELS].to_numpy('float32')
print(f'Q_CTX  {X_CTX.shape[1]} cot | Q_FULL {X_FULL.shape[1]} cot '
      f'(rieng gia tri: {X_FULL.shape[1]-X_CTX.shape[1]})')

# ---- COÅNG: Q_CTX khong duoc chua KET QUA lam sang (§2.3) ----
# assert tren TEN CHINH XAC (khong nham voi ten xet nghiem nhu "inorganic phosphate");
# them mot canh bao mem theo chuoi con de bat truong hop phat sinh.
_FORBID_CTX = {'ctx_status', 'ctx_recency_bin', 'cnt_t4_n_prior_positive_ever',
               'm_rec_t1_days_since_last_positive', 'm_rec_t3_days_since_last_positive'}
_BAD = sorted(_FORBID_CTX & set(NM_C))
assert not _BAD, f'Q_CTX chua feature mang KET QUA lam sang: {_BAD}'
_WARN = [n for n in NM_C if any(k in n.lower() for k in ('positive', 'status', 'recency'))]
if _WARN: print(f'  ! xem lai — ten CTX co ve mang ket qua: {_WARN}')
assert set(NM_C).isdisjoint(NM_F), f'trung ten CTX/FULL: {set(NM_C) & set(NM_F)}'
_grp = {}
for n in NM_C:
    _grp.setdefault(n.split('_')[0], []).append(n)
print('  Q_CTX theo nhom: ' + ' | '.join(f'{k} {len(v)}' for k, v in _grp.items()))
print('    demo=demographics  m=missingness(lab/vit/rec)  cnt=dem do luong  '
      'bin=co dat lenh  ctx=noi cham soc')
print('  -> DA CHUYEN sang Q_FULL vi mang KET QUA: t1_status, t4_n_prior_positive_ever,')
print('     t3_recency_bin, mask cua t1/t3_days_since_last_positive.')

# do day EHR — dung cho phan tang hoan vi o C6 (§6.3)
_mcols = [i for i, n in enumerate(NM_C) if n.startswith('m_')]
COMPLETE = X_CTX[:, _mcols].mean(1)

try:
    from xgboost import XGBClassifier
    MK = lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08, subsample=0.8,
                               colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, random_state=42,
                               eval_metric='logloss', tree_method='hist')
    ENGINE = 'xgboost'
except Exception:
    from sklearn.ensemble import HistGradientBoostingClassifier
    MK = lambda: HistGradientBoostingClassifier(max_iter=300, max_depth=6, learning_rate=0.08,
                                                random_state=42)
    ENGINE = 'sklearn HistGradientBoosting'
print(f'engine EHR: {ENGINE}  (§2.3: khong dung deep EHR encoder trong primary)')

def _rows(sp):
    m = (ehr_f.split == sp).to_numpy()
    o = np.argsort(ehr_f.loc[m, 'study_id'].to_numpy(), kind='stable')
    return np.where(m)[0][o]
Q = {}
for tag, X in (('ctx', X_CTX), ('full', X_FULL)):
    for sp in ('val', 'test'):
        p = f'{SC_DIR}/predictions/ehr_{tag}_probs_{sp}.npz'
        if os.path.exists(p): Q[(tag, sp)] = np.load(p)['q']; continue
        if sp == 'test' and not globals().get('RUN_TEST', False):
            print(f'  ehr_{tag}_{sp}: bo qua (RUN_TEST = False)'); continue
        r = _rows(sp); q = np.zeros((len(r), C), 'float32'); t0 = time.time()
        for j in tqdm(range(C), desc=f'Q_{tag.upper()} -> {sp}'):
            q[:, j] = MK().fit(X[tr_m], Y_ALL[tr_m, j]).predict_proba(X[r])[:, 1]
        np.savez_compressed(p, q=q); Q[(tag, sp)] = q
        print(f'  Q_{tag.upper()} {sp}: {(time.time()-t0)/60:.1f} phut')
for (tag, sp), q in Q.items():
    assert q.shape == (len(SPL[sp]['sid']), C), (tag, sp, q.shape)
    assert np.array_equal(np.sort(ehr_f.loc[(ehr_f.split == sp).to_numpy(), 'study_id'].to_numpy()),
                          SPL[sp]['sid']), f'thu tu study {sp} khong khop'
print()
print('=== TRAN EHR-ONLY tren VAL (AUROC; AP phu thuoc prevalence nen kho so giua nhom) ===')
_yv = SPL['val']['y']
_t = pd.DataFrame(dict(label=LABELS, tier=TIER,
                       AUROC_ctx=[roc_auc_score(_yv[:, j], Q[('ctx', 'val')][:, j]) for j in range(C)],
                       AUROC_full=[roc_auc_score(_yv[:, j], Q[('full', 'val')][:, j]) for j in range(C)]))
_t['gia_tri_them'] = _t.AUROC_full-_t.AUROC_ctx
print(_t.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'  trung binh: CTX {_t.AUROC_ctx.mean():.4f} | FULL {_t.AUROC_full.mean():.4f} | '
      f'gia tri them {_t.gia_tri_them.mean():+.4f}')

demographics tim thay: KHONG CO trong cohort
  ! §2.3 liet ke demographics cho Q_CTX nhung cohort khong co cot nao.
    Q_CTX chay khong co demographics — ghi vao bao cao nhu mot sai lech so voi plan.
Q_CTX  22 cot | Q_FULL 170 cot (rieng gia tri: 148)
  Q_CTX theo nhom: m 14 | cnt 4 | bin 3 | ctx 1
    demo=demographics  m=missingness(lab/vit/rec)  cnt=dem do luong  bin=co dat lenh  ctx=noi cham soc
  -> DA CHUYEN sang Q_FULL vi mang KET QUA: t1_status, t4_n_prior_positive_ever,
     t3_recency_bin, mask cua t1/t3_days_since_last_positive.
engine EHR: xgboost  (§2.3: khong dung deep EHR encoder trong primary)
  ehr_ctx_test: bo qua (RUN_TEST = False)
  ehr_full_test: bo qua (RUN_TEST = False)

=== TRAN EHR-ONLY tren VAL (AUROC; AP phu thuoc prevalence nen kho so giua nhom) ===
                     label   tier  AUROC_ctx  AUROC_full  gia_tri_them
               Atelectasis   head     0.6629      0.6859        0.0229
              Cardiomegaly   head     0.7018      0.7329        0.031

## C4 · Bộ đo + khung nested CV

`ap_grouped` verbatim từ nb16 `C15` (đã kiểm khớp `sklearn` bit-for-bit kể cả khi có tie).

Bootstrap §11.1: `2,000` resample, cluster `subject_id`, **paired across arms** — cùng một bộ
resample cho mọi arm, nếu không thì hiệu số giữa arm sẽ mang thêm nhiễu không cần thiết.

`valid fraction` báo cáo riêng cho nhãn hiếm (§11.1).

### Chọn `λ`

§7.2 chỉ nói không được fit hyperparameter trên outer-held-out. Không nói chọn theo chỉ số nào.
Dùng **macro mAP trên inner-held-out** — trùng endpoint §9.1. Ghi rõ vì đây là quyết định của
tôi, không phải của plan.

In [5]:
import torch
TD = torch.float64
def _prep(p):
    o = np.argsort(-p, kind='stable'); ps = p[o]
    gid = np.r_[0, np.cumsum(ps[1:] != ps[:-1])].astype('int64')
    return o, gid, int(gid[-1])+1
def ap_grouped(yo, co, gid, ng):
    a = np.bincount(gid, weights=yo*co, minlength=ng)
    n = np.bincount(gid, weights=co, minlength=ng)
    npos = a.sum()
    if npos <= 0: return np.nan
    TP, N = np.cumsum(a), np.cumsum(n); m = a > 0
    return float((a[m]/npos*(TP[m]/N[m])).sum())
def ap_vec(P, y):
    out = np.full(C, np.nan)
    for j in range(C):
        if y[:, j].sum() == 0: continue
        o, gid, ng = _prep(P[:, j])
        out[j] = ap_grouped(y[o, j].astype('float64'), np.ones(len(o)), gid, ng)
    return out
_chk = ap_vec(A0_new, SPL['val']['y'])
_ref = np.array([average_precision_score(SPL['val']['y'][:, j], A0_new[:, j]) for j in range(C)])
assert np.nanmax(np.abs(_chk-_ref)) < 1e-9, 'ap_grouped lech sklearn'
print('ap_grouped == sklearn  OK')
gm = lambda ap: {g: float(np.nanmean(ap[jj])) for g, jj in GRP.items()}

# ---------------- bootstrap paired, cluster = subject ----------------
_su = np.unique(V['subject']); _six = np.searchsorted(_su, V['subject']); _nsu = len(_su)
_rg = np.random.default_rng(20240301)
BOOT_C = [np.bincount(_rg.integers(0, _nsu, _nsu), minlength=_nsu)[_six] for _ in range(N_BOOT)]
def boot_groups(P, y):
    pr = [_prep(P[:, j]) for j in range(C)]
    yo = [y[pr[j][0], j].astype('float64') for j in range(C)]
    out = {g: np.empty(N_BOOT) for g in GRP}; valid = np.zeros(C)
    for b, c in enumerate(BOOT_C):
        a = np.array([ap_grouped(yo[j], c[pr[j][0]].astype('float64'), pr[j][1], pr[j][2])
                      for j in range(C)])
        valid += ~np.isnan(a)
        for g, jj in GRP.items(): out[g][b] = np.nanmean(a[jj])
    return out, valid/N_BOOT
def dci(bA, bB, apA, apB, g):
    d = bA[g]-bB[g]; lo, hi = np.percentile(d, [2.5, 97.5])
    return dict(delta=float(np.nanmean(apA[GRP[g]])-np.nanmean(apB[GRP[g]])),
                lo=float(lo), hi=float(hi), excl0=bool(lo > 0 or hi < 0))
print(f'bootstrap {N_BOOT} resample cluster=subject, PAIRED (dung chung cho moi arm)')

def nested_cv(fit_fn, pred_fn, grid, y, outer, inner, tag=''):
    """fit_fn(idx, lam) -> par ; pred_fn(par, idx) -> [len(idx), C]. Tra OOF + lambda tung fold."""
    n = len(y); OOF = np.full((n, C), np.nan); seen = np.zeros(n, 'int64'); rec = []
    for k in range(K_OUTER):
        tr = np.where(outer != k)[0]; te = np.where(outer == k)[0]
        if len(grid) == 1:
            best = grid[0]
        else:
            sc = []
            for lam in grid:
                s = []
                for j in range(K_INNER):
                    itr = tr[inner[tr] != j]; ite = tr[inner[tr] == j]
                    s.append(float(np.nanmean(ap_vec(pred_fn(fit_fn(itr, lam), ite), y[ite]))))
                sc.append(np.mean(s))
            best = grid[int(np.argmax(sc))]
        par = fit_fn(tr, best)
        OOF[te] = pred_fn(par, te); seen[te] += 1
        rec.append(dict(fold=k, lam=list(best) if isinstance(best, tuple) else best,
                        **{f'p_{i}': float(v) for i, v in enumerate(np.atleast_1d(par['flat']))}))
    assert (seen == 1).all() and np.isfinite(OOF).all(), f'{tag}: OOF loi'
    return OOF, pd.DataFrame(rec)
print('nested_cv ready | lambda chon bang macro mAP tren INNER-held-out (quyet dinh cua toi)')

ap_grouped == sklearn  OK
bootstrap 2000 resample cluster=subject, PAIRED (dung chung cho moi arm)
nested_cv ready | lambda chon bang macro mAP tren INNER-held-out (quyet dinh cua toi)


## C5 · View arms — `A0` · `AG` · `A1` · `AV` · `AS`

| arm | tham số | câu hỏi |
|---|---:|---|
| `A0` | 0 | mốc cố định `0.7/0.3` |
| `AG` | 1 (`a₀`) | `0.7` có tối ưu **toàn cục** không? |
| `A1` | 14 (`a₀ + Δ_c`) | trọng số **theo bệnh** có hơn trọng số chung không? |
| `AV` | 13 (`γ_c`) | gain đến từ **nội dung** lateral hay chỉ từ **việc có** lateral? |
| `AS` | 14 × 5 perm | `A1` trên lateral **đã xáo** — nếu bằng nhau thì không được claim khai thác nội dung |

`α = 0.5 + 0.5·σ(a)` ⇒ `α ∈ [0.5, 1)`. Init `α = 0.70` ⇒ `σ(a) = 0.40` ⇒ `a = logit(0.4)`.

Về mặt toán học `α = 1.0` chỉ đạt ở giới hạn `a → ∞`, nhưng trong `float64` thì `σ(a) = 1.0`
**đúng bằng** khi `a ≳ 37`, nên ràng buộc `0.50 ≤ α ≤ 1.00` của plan **thoả được cả hai đầu**.

Vấn đề thật không phải ràng buộc mà là **bão hoà**: ở hai biên `∂α/∂a → 0`, LBFGS có thể đứng
lại. `λ_d` kéo `Δ_c` về 0 nên hạn chế được, nhưng cell vẫn **đếm số (fold, nhãn) chạm biên** —
nghiệm biên nghĩa là ràng buộc đang **có hiệu lực**, và khi đó khoảng `[0.5, 1]` (thay vì
`[0, 1]`) thực sự đang định hình kết quả. Đó chính là thứ §3.3 dành `α ∈ [0,1]` làm sensitivity.

`AS` hoán vị **ở mức subject, trong cùng care setting** (§4.2): mỗi bệnh nhân nhận lateral từ
một bệnh nhân hiến khác cùng setting. `AS` được **nested CV đầy đủ** như `A1` — cho control ít
ưu đãi hơn `A1` sẽ làm phép so `A1 > AS` thiên vị.

**Chi phí:** `A1` cần `5 outer × 4 inner × 16 λ + 5 = 325` lần LBFGS; `AS` nhân 5. Đo thử ở
`n = 500`: `0.16 s`/fit ⇒ ước **3–6 phút** cho `A1`, **15–30 phút** cho `AS` ở quy mô VAL, kể cả
CPU-only.

In [6]:
LG = lambda p: np.log(np.clip(p, 1e-6, 1-1e-6)/(1-np.clip(p, 1e-6, 1-1e-6)))
A_INIT = math.log(ALPHA_INIT*2-1) - math.log(1-(ALPHA_INIT*2-1))     # logit(0.40)
assert abs(0.5+0.5/(1+math.exp(-A_INIT)) - ALPHA_INIT) < 1e-12

def _z_view(mode, par, PF, PL, MF, ML):
    if mode == 'AV':
        zF, zL = torch.logit(PF.clamp(1e-6, 1-1e-6)), torch.logit(PL.clamp(1e-6, 1-1e-6))
        return torch.where(MF[:, None] > 0, zF + par['g'][None, :]*ML[:, None], zL)
    if mode == 'A0':  al = torch.full((C,), W_F0, dtype=TD, device=dev)
    elif mode == 'AG': al = (0.5+0.5*torch.sigmoid(par['a0'])).expand(C)
    else:              al = 0.5+0.5*torch.sigmoid(par['a0']+par['d'])
    num = MF[:, None]*al[None, :]*PF + ML[:, None]*(1-al)[None, :]*PL
    den = MF[:, None]*al[None, :] + ML[:, None]*(1-al)[None, :]
    return torch.logit((num/den.clamp(min=1e-00)).clamp(1e-6, 1-1e-6))

def make_view_fitter(pF, pL, mF, mL, y, mode):
    T = lambda x: torch.as_tensor(x, dtype=TD, device=dev)
    PF, PL, MF, ML, Y = T(pF), T(pL), T(mF), T(mL), T(y)
    def fit(idx, lam):
        lg, ld = (lam if isinstance(lam, tuple) else (0.0, lam))
        a0 = torch.tensor(A_INIT, dtype=TD, device=dev, requires_grad=mode in ('AG', 'A1'))
        d  = torch.zeros(C, dtype=TD, device=dev, requires_grad=mode == 'A1')
        g  = torch.zeros(C, dtype=TD, device=dev, requires_grad=mode == 'AV')
        par = dict(a0=a0, d=d, g=g)
        ps = [p for p in (a0, d, g) if p.requires_grad]
        if ps:
            opt = torch.optim.LBFGS(ps, max_iter=MAX_ITER, line_search_fn='strong_wolfe',
                                    tolerance_grad=1e-10, tolerance_change=1e-12)
            i = torch.as_tensor(idx, device=dev)
            def closure():
                opt.zero_grad()
                z = _z_view(mode, par, PF[i], PL[i], MF[i], ML[i])
                L = torch.nn.functional.binary_cross_entropy_with_logits(z, Y[i])
                if mode in ('AG', 'A1'): L = L + lg*(a0-A_INIT)**2
                if mode == 'A1':         L = L + ld*(d**2).sum()
                if mode == 'AV':         L = L + ld*(g**2).sum()
                L.backward(); return L
            opt.step(closure)
        flat = torch.cat([a0.reshape(1), d, g]).detach().cpu().numpy()
        return dict(a0=a0.detach(), d=d.detach(), g=g.detach(), flat=flat, lam=lam)
    def pred(par, idx):
        i = torch.as_tensor(idx, device=dev)
        with torch.no_grad():
            return torch.sigmoid(_z_view(mode, par, PF[i], PL[i], MF[i], ML[i])).cpu().numpy()
    return fit, pred

def shuffle_lateral(pL, subject, setting, mF, mL, seed):
    out = pL.copy(); r = np.random.default_rng(seed)
    both = np.where((mF > 0) & (mL > 0))[0]
    for st in np.unique(setting[both]):
        idx = both[setting[both] == st]
        subs = np.unique(subject[idx])
        if len(subs) < 2: continue
        donor = dict(zip(subs, subs[r.permutation(len(subs))]))
        bysub = {s: idx[subject[idx] == s] for s in subs}
        for i in idx: out[i] = pL[r.choice(bysub[donor[subject[i]]])]
    return out

GRID = dict(AG=[(g, 0.0) for g in LAMBDA_G],
            A1=[(g, d) for g in LAMBDA_G for d in LAMBDA_D],
            AV=[(0.0, d) for d in LAMBDA_D])
VIEW_OOF, VIEW_PAR = {}, {}
for s in HAVE:
    z = PFL[(s, 'val')]; pF, pL = z['pF'], z['pL']
    VIEW_OOF[(s, 'A0')] = pool_alpha(pF, pL, V['mF'], V['mL'], np.full(C, W_F0))
    for mode in ('AG', 'A1', 'AV'):
        f, p = make_view_fitter(pF, pL, V['mF'], V['mL'], V['y'], mode)
        t0 = time.time()
        oof, rec = nested_cv(f, p, GRID[mode], V['y'], V['outer'], V['inner'], f'{mode}/s{s}')
        VIEW_OOF[(s, mode)] = oof; VIEW_PAR[(s, mode)] = rec
        print(f'  seed {s} {mode}: {time.time()-t0:.0f}s | lambda tung fold {list(rec.lam)}')
    # ---- AS: A1 tren lateral da xao, 5 seed hoan vi khoa truoc ----
    for ps_ in PERM_SEEDS:
        pLs = shuffle_lateral(pL, V['subject'], V['setting'], V['mF'], V['mL'], ps_)
        f, p = make_view_fitter(pF, pLs, V['mF'], V['mL'], V['y'], 'A1')
        VIEW_OOF[(s, f'AS{ps_}')] = nested_cv(f, p, GRID['A1'], V['y'], V['outer'],
                                              V['inner'], f'AS{ps_}/s{s}')[0]
    print(f'  seed {s} AS: xong {len(PERM_SEEDS)} hoan vi')

# ---------------- COÅNG §8 ----------------
_s0 = HAVE[0]
_f, _p = make_view_fitter(PFL[(_s0, 'val')]['pF'], PFL[(_s0, 'val')]['pL'],
                          V['mF'], V['mL'], V['y'], 'A1')
_p0 = _p(dict(a0=torch.tensor(A_INIT, dtype=TD, device=dev), d=torch.zeros(C, dtype=TD, device=dev),
              g=torch.zeros(C, dtype=TD, device=dev)), np.arange(len(V['sid'])))
_e = float(np.abs(LG(_p0)-LG(VIEW_OOF[(_s0, 'A0')])).max())
print()
print(f'COÅNG §8 · A1 tai init == A0 (logit): max_abs_diff {_e:.2e} {"OK" if _e <= 1e-8 else "FAIL"}')
assert _e <= 1e-8
print(f'COÅNG §8 · alpha thuc te thuoc [0.5, 1) — plan ghi [0.50, 1.00], can tren chi tiem can')
_al = {s: 0.5+0.5/(1+np.exp(-(VIEW_PAR[(s, 'A1')][[f'p_{i}' for i in range(1, C+1)]].to_numpy()
                              + VIEW_PAR[(s, 'A1')]['p_0'].to_numpy()[:, None]))) for s in HAVE}
_A = pd.concat([pd.DataFrame(v, columns=LABELS).assign(seed=s, fold=range(K_OUTER))
                for s, v in _al.items()])
_A.to_csv(f'{SC_DIR}/metrics/alpha_stability.csv', index=False)
print()
print('=== alpha_c hoc duoc (trung binh qua fold/seed; 0.70 la init) ===')
print(_A[LABELS].mean().to_frame('alpha_tb').assign(
    lech_init=lambda t: t.alpha_tb-ALPHA_INIT,
    sd_qua_fold=_A[LABELS].std()).to_string(float_format=lambda v: f'{v:.4f}'))
_v = _A[LABELS].to_numpy()
_lo = float((np.abs(_v-ALPHA_LO) < 1e-6).mean()); _hi = float((np.abs(_v-1.0) < 1e-6).mean())
print()
print(f'  cham bien: alpha = {ALPHA_LO} o {_lo:.1%} (fold x nhan) | alpha = 1.0 o {_hi:.1%}')
if _lo+_hi > 0.10:
    print(f'  ! {_lo+_hi:.0%} nghiem nam O BIEN -> rang buoc [0.5, 1] DANG dinh hinh ket qua.')
    print(f'    §3.3 co san sensitivity alpha thuoc [0,1]; chay no de biet bi cat bao nhieu.')
    print(f'    Nhung §3.3 cam dung sensitivity do de CUU mot primary null.')
else:
    print(f'  nghiem gan nhu deu nam TRONG khoang -> rang buoc khong phai yeu to quyet dinh.')

  seed 42 AG: 10s | lambda tung fold [[0.01, 0.0], [0.01, 0.0], [0.01, 0.0], [0.01, 0.0], [0.01, 0.0]]
  seed 42 A1: 84s | lambda tung fold [[0.01, 0.1], [0.01, 0.01], [0.01, 0.01], [0.01, 0.01], [0.01, 0.01]]
  seed 42 AV: 2s | lambda tung fold [[0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0]]
  seed 42 AS: xong 5 hoan vi

COÅNG §8 · A1 tai init == A0 (logit): max_abs_diff 6.22e-15 OK
COÅNG §8 · alpha thuc te thuoc [0.5, 1) — plan ghi [0.50, 1.00], can tren chi tiem can

=== alpha_c hoc duoc (trung binh qua fold/seed; 0.70 la init) ===
                            alpha_tb  lech_init  sd_qua_fold
Atelectasis                   0.7044     0.0044       0.0004
Cardiomegaly                  0.7043     0.0043       0.0003
Consolidation                 0.7040     0.0040       0.0002
Edema                         0.7043     0.0043       0.0003
Enlarged_Cardiomediastinum    0.7039     0.0039       0.0002
Fracture                      0.7044     0.0044       0.0003
Lung_Lesion        

## C6 · EHR residual — `B0` · `B1` · `B1_CTX` · `B1_SHUF`

$$z^{\text{FINAL}}_{i,c}=z^{\text{IMG}}_{i,c}+\beta_c\,u^{\text{EHR}}_{i,c},
\qquad \beta_c=\sigma(b_c),\; b_c^{(0)}=-8$$

Hệ số của ảnh **cố định bằng 1** (§5.1) — không học lại. `u` chuẩn hoá theo lớp bằng `μ_c, σ_c`
tính **chỉ trên phần calibration-training của fold** (§5.2). Không có intercept riêng (§5.3).

### Tuần tự, không joint (§5.4)

Trong **mỗi** outer fold: fit `α` trên outer-train → **đóng băng** → fit `β` trên outer-train →
dự đoán outer-held-out.

Để chọn `λ_β` bằng inner CV mà không rò rỉ, `α` được fit **lại trên inner-train** ở mỗi inner
fold. Nếu dùng `α` của outer-train cho vòng inner thì `α` đã nhìn thấy inner-held-out, và
`λ_β` được chọn trên một ước lượng lạc quan.

### ⚠ `β` init không **bằng** 0

`σ(−8) = 3.35e-4`, không phải `0`. Nên assertion §8 *"B0/B1 init logits == image-only arm"*
**không** đúng theo nghĩa đẳng thức. Cell kiểm phiên bản có nghĩa: tại init, **thứ hạng** giữ
nguyên (Spearman `= 1`) và `|ΔAP| < 1e-5`.

### `B1_SHUF` (§6.3)

Hoán vị `q_FULL` ở **mức subject**, trong cùng **care setting** *và* cùng **ngũ phân vị độ đầy
EHR**. 5 seed khoá trước. `B1 > A1` một mình chưa đủ — nhánh residual có thể học artefact
cohort/context.

In [7]:
def _u(q, idx_fit, idx_all):
    lq = LG(q); mu = lq[idx_fit].mean(0); sd = lq[idx_fit].std(0)
    return (lq[idx_all]-mu)/(sd+1e-8)

def fit_beta(z_tr, u_tr, y_tr, lam):
    T = lambda x: torch.as_tensor(x, dtype=TD, device=dev)
    Z, U, Y = T(z_tr), T(u_tr), T(y_tr)
    b = torch.full((C,), B_INIT, dtype=TD, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([b], max_iter=MAX_ITER, line_search_fn='strong_wolfe',
                            tolerance_grad=1e-10, tolerance_change=1e-12)
    def cl():
        opt.zero_grad()
        be = torch.sigmoid(b)
        L = torch.nn.functional.binary_cross_entropy_with_logits(Z+be[None, :]*U, Y) \
            + lam*(be**2).sum()
        L.backward(); return L
    opt.step(cl)
    return torch.sigmoid(b).detach().cpu().numpy()

def shuffle_ehr(q, subject, setting, complete, seed):
    out = q.copy(); r = np.random.default_rng(seed)
    quint = pd.qcut(pd.Series(complete), 5, labels=False, duplicates='drop').to_numpy()
    key = pd.Series([f'{a}|{b}' for a, b in zip(setting, quint)])
    for _, ix in key.groupby(key).groups.items():
        idx = np.asarray(list(ix))
        subs = np.unique(subject[idx])
        if len(subs) < 2: continue
        donor = dict(zip(subs, subs[r.permutation(len(subs))]))
        bysub = {s: idx[subject[idx] == s] for s in subs}
        for i in idx: out[i] = q[r.choice(bysub[donor[subject[i]]])]
    return out

def nested_ehr(pF, pL, q, view_mode, lam_view_per_fold, tag=''):
    n = len(V['sid']); OOF = np.full((n, C), np.nan); rec = []
    fitV, predV = make_view_fitter(pF, pL, V['mF'], V['mL'], V['y'], view_mode)
    for k in range(K_OUTER):
        tr = np.where(V['outer'] != k)[0]; te = np.where(V['outer'] == k)[0]
        lamV = lam_view_per_fold[k]
        sc = []
        for lb in LAMBDA_B:
            s_ = []
            for j in range(K_INNER):
                itr = tr[V['inner'][tr] != j]; ite = tr[V['inner'][tr] == j]
                pv = fitV(itr, lamV)                       # alpha fit tren INNER-train
                zi = LG(predV(pv, itr)); zo = LG(predV(pv, ite))
                ui = _u(q, itr, itr); uo = _u(q, itr, ite)
                be = fit_beta(zi, ui, V['y'][itr], lb)
                s_.append(float(np.nanmean(ap_vec(1/(1+np.exp(-(zo+be[None, :]*uo))), V['y'][ite]))))
            sc.append(np.mean(s_))
        lb = LAMBDA_B[int(np.argmax(sc))]
        pv = fitV(tr, lamV)
        ztr, zte = LG(predV(pv, tr)), LG(predV(pv, te))
        utr, ute = _u(q, tr, tr), _u(q, tr, te)
        be = fit_beta(ztr, utr, V['y'][tr], lb)
        OOF[te] = 1/(1+np.exp(-(zte+be[None, :]*ute)))
        rec.append(dict(fold=k, lam_beta=lb, **{f'beta_{l}': float(v) for l, v in zip(LABELS, be)}))
    assert np.isfinite(OOF).all(), f'{tag}: OOF loi'
    return OOF, pd.DataFrame(rec)

EHR_OOF, BETA, QSHUF = {}, {}, {}
_lamA0 = [(0.0, 0.0)]*K_OUTER
for s in HAVE:
    z = PFL[(s, 'val')]; pF, pL = z['pF'], z['pL']
    _lamA1 = [tuple(v) if isinstance(v, list) else v for v in VIEW_PAR[(s, 'A1')].lam]
    for arm, vm_, qq, lam in (('B0', 'A0', Q[('full', 'val')], _lamA0),
                              ('B1', 'A1', Q[('full', 'val')], _lamA1),
                              ('B1_CTX', 'A1', Q[('ctx', 'val')], _lamA1)):
        t0 = time.time()
        EHR_OOF[(s, arm)], BETA[(s, arm)] = nested_ehr(pF, pL, qq, vm_, lam, f'{arm}/s{s}')
        print(f'  seed {s} {arm}: {time.time()-t0:.0f}s | lambda_beta {list(BETA[(s,arm)].lam_beta)}')
    for ps_ in PERM_SEEDS:
        qs = shuffle_ehr(Q[('full', 'val')], V['subject'], V['setting'], COMPLETE[_rows('val')], ps_)
        QSHUF[ps_] = qs                       # cong §8 dung lai, khong hoan vi lai
        EHR_OOF[(s, f'B1_SHUF{ps_}')] = nested_ehr(pF, pL, qs, 'A1', _lamA1, f'SHUF{ps_}')[0]
    print(f'  seed {s} B1_SHUF: xong {len(PERM_SEEDS)} hoan vi')

# ================= COÅNG §8 · AMENDMENT 2026-07-31 =================
# Plan dong thoi yeu cau b_c = -8 (=> beta = 3.354e-04) VA "init logits == image-only".
# Hai dieu do KHONG the cung dung. Thay exact-identity bang NEAR-IDENTITY khai bao truoc.
# Nhieu init KHONG common-mode giua cac B-arm: B0 nam tren A0, B1 tren A1, B1_CTX dung
# u_CTX, B1_SHUF dung u da hoan vi -> ranking nen va vector u khac nhau. Vi vay moi arm
# phai kiem voi ĐUNG paired image baseline cua no.
RHO_INIT_MIN,  DAP_INIT_MAX  = 0.9999,  1e-4      # 1e-4 = 5% cua cong thuc dung +0.002
RHO_SCALE_MIN, DAP_SCALE_MAX = 0.99999, 1e-5
B_SCALE_CHECK       = -12.0                        # DIAGNOSTIC ONLY — khong doi init chinh
PRIMARY_EFFECT_GATE = 2e-3

_sp = lambda u, v: float(np.corrcoef(pd.Series(u).rank(), pd.Series(v).rank())[0, 1])
LG_CEIL = float(LG(np.array([1.0]))[0])            # tran do LG kep o 1-1e-6

def paired_init(pF, pL, q, view_mode, lam_view_per_fold):
    """z_img va u theo DUNG cau truc fold cua nested_ehr (u chuan hoa tren outer-train)."""
    n = len(V['sid']); Z = np.full((n, C), np.nan); U = np.full((n, C), np.nan)
    fitV, predV = make_view_fitter(pF, pL, V['mF'], V['mL'], V['y'], view_mode)
    for k in range(K_OUTER):
        tr = np.where(V['outer'] != k)[0]; te = np.where(V['outer'] == k)[0]
        pv = fitV(tr, lam_view_per_fold[k])
        Z[te] = LG(predV(pv, te)); U[te] = _u(q, tr, te)
    assert np.isfinite(Z).all() and np.isfinite(U).all()
    return Z, U

def init_diag(Z, U, b):
    be = 1/(1+math.exp(-b))
    P0, Pb = 1/(1+np.exp(-Z)), 1/(1+np.exp(-(Z+be*U)))
    a0, ab = ap_vec(P0, V['y']), ap_vec(Pb, V['y'])
    d = np.abs(ab-a0)
    return dict(beta=be, P0=P0, Pb=Pb,
                rho_min=min(_sp(Pb[:, j], P0[:, j]) for j in range(C)),
                dap_max=float(np.nanmax(d)), worst=LABELS[int(np.nanargmax(d))],
                grp={g: float(abs(np.nanmean(ab[ix])-np.nanmean(a0[ix])))
                     for g, ix in GRP.items()})

_s = HAVE[0]
_z = PFL[(_s, 'val')]; _pF, _pL = _z['pF'], _z['pL']
_lamA1s = [tuple(v) if isinstance(v, list) else v for v in VIEW_PAR[(_s, 'A1')].lam]
_ps0 = PERM_SEEDS[0]
ARMS_INIT = (('B0',                'A0', Q[('full', 'val')], _lamA0),
             ('B1',                'A1', Q[('full', 'val')], _lamA1s),
             ('B1_CTX',            'A1', Q[('ctx',  'val')], _lamA1s),
             (f'B1_SHUF{_ps0}',    'A1', QSHUF[_ps0],        _lamA1s))

print()
print('=== PHAN BO |u| (giai thich do lon cua nhieu init) ===')
_UST = {}
for _arm, _vm, _q, _lam in ARMS_INIT:
    _Z, _U = paired_init(_pF, _pL, _q, _vm, _lam)
    _UST[_arm] = (_Z, _U, _vm)
    _au = np.abs(_U[np.isfinite(_U)])
    _b8 = 1/(1+math.exp(-B_INIT))
    print(f'  {_arm:<12} baseline {_vm} | p50 {np.quantile(_au,.5):5.2f}  p90 {np.quantile(_au,.9):5.2f}  '
          f'p99 {np.quantile(_au,.99):6.2f}  p999 {np.quantile(_au,.999):7.2f}  max {_au.max():8.2f}'
          f'  -> max|dz| = beta*max|u| = {_b8*_au.max():.2e}')
print()
print('=== |u| tung nhan (arm B1 = Q_FULL vs B1_CTX = Q_CTX) ===')
_rows_u = []
for _arm in ('B1', 'B1_CTX'):
    _U = _UST[_arm][1]
    for j, l in enumerate(LABELS):
        _a = np.abs(_U[:, j])
        _rows_u.append(dict(arm=_arm, label=l, tier=TIER[j], p50=np.quantile(_a, .5),
                            p99=np.quantile(_a, .99), p999=np.quantile(_a, .999), mx=_a.max()))
_UT = pd.DataFrame(_rows_u)
print(_UT.pivot(index='label', columns='arm', values=['p99', 'p999', 'mx'])
        .to_string(float_format=lambda v: f'{v:.2f}'))

print()
print(f'=== COÅNG §8 · near-identity tai init (b = {B_INIT}, beta = {1/(1+math.exp(-B_INIT)):.3e}) ===')
INIT_DIAG = {}
for _arm, _vm, _q, _lam in ARMS_INIT:
    _Z, _U, _ = _UST[_arm]
    d8 = init_diag(_Z, _U, B_INIT)
    # Duong code: |LG(Pb) - LG(P0) - beta*u| KHONG duoc vuot qua chinh nhieu beta*max|u|.
    # KHONG assert = 0: LG kep o [1e-6, 1-1e-6] nen o cac muc bao hoa dang thuc vo — do la
    # giai tao cua PHEP KIEM, khong phai cua duong model (nested_ehr tra ve XAC SUAT, khong
    # LG lai). Nhung neu lo dung FITTED beta thay vi beta_init, hoac lech hang / lech u,
    # residual se VUOT qua beta_init*max|u| -> day moi la thu can bat.
    _dz = d8['beta']*float(np.abs(_U).max())
    _res = float(np.abs((LG(d8['Pb'])-LG(d8['P0']))-d8['beta']*_U).max())
    _sat = float((np.abs(LG(d8['P0'])) >= LG_CEIL-1e-9).mean())
    assert _res <= _dz*(1+1e-6)+1e-12, (
        f'{_arm}: residual {_res:.2e} > nhieu beta_init*max|u| = {_dz:.2e}. '
        'Kiem: dung nham FITTED beta thay vi init? lech hang du doan? u sai arm?')
    # paired baseline phai trung VIEW_OOF cua chinh arm do (bat lech fold giua C5 va C6)
    _ref = VIEW_OOF[(_s, _vm)]
    _dv = float(np.abs(d8['P0']-_ref).max())
    assert _dv < 1e-8, f'{_arm}: baseline anh-thuan lech VIEW_OOF[{_vm}] {_dv:.2e} -> fold khong khop'
    d12 = init_diag(_Z, _U, B_SCALE_CHECK)
    # thang tuyen tinh: delta12 == (beta12/beta8) * delta8
    _rt = d12['beta']/d8['beta']
    assert np.allclose(d12['beta']*_U, _rt*(d8['beta']*_U), rtol=1e-6, atol=1e-12)
    INIT_DIAG[_arm] = dict(baseline=_vm, beta8=d8['beta'], rho8=d8['rho_min'],
                           dap8_max=d8['dap_max'], worst8=d8['worst'], grp8=d8['grp'],
                           beta12=d12['beta'], rho12=d12['rho_min'], dap12_max=d12['dap_max'],
                           frac_gate=d8['dap_max']/PRIMARY_EFFECT_GATE,
                           max_abs_u=float(np.abs(_U).max()), resid=_res)
    print(f'  {_arm:<12} vs {_vm}')
    print(f'      b={B_INIT:<5} rho_min {d8["rho_min"]:.8f}  max|dAP| {d8["dap_max"]:.2e} '
          f'({d8["dap_max"]/PRIMARY_EFFECT_GATE:5.2%} cua cong {PRIMARY_EFFECT_GATE:g})  '
          f'nhan te nhat {d8["worst"]}')
    print('      |dAP| nhom: ' + ' | '.join(f'{g} {v:.2e}' for g, v in d8['grp'].items()))
    _shr = d8['dap_max']/max(d12['dap_max'], 1e-30)
    print(f'      b={B_SCALE_CHECK:<5} rho_min {d12["rho_min"]:.8f}  max|dAP| {d12["dap_max"]:.2e}'
          f'   (beta nho hon {1/_rt:.0f} lan -> |dAP| nho hon {_shr:.0f} lan)')
    print(f'      bao hoa: {_sat:.2%} muc co |z_img| cham tran LG {LG_CEIL:.4f}  |  '
          f'residual vong LG {_res:.2e} <= beta*max|u| {_dz:.2e}')
    assert d8['rho_min'] >= RHO_INIT_MIN, f'{_arm}: Spearman {d8["rho_min"]:.8f} < {RHO_INIT_MIN}'
    assert d8['dap_max'] <= DAP_INIT_MAX, f'{_arm}: max|dAP| {d8["dap_max"]:.2e} > {DAP_INIT_MAX:g}'
    assert d12['rho_min'] >= RHO_SCALE_MIN and d12['dap_max'] <= DAP_SCALE_MAX, (
        f'{_arm}: b={B_SCALE_CHECK} cho rho {d12["rho_min"]:.8f} / dAP {d12["dap_max"]:.2e} '
        f'(nguong {RHO_SCALE_MIN} / {DAP_SCALE_MAX:g}). |dAP| nho hon {_shr:.1f} lan trong '
        f'khi beta nho hon {1/_rt:.0f} lan.  ==> Neu |dAP| KHONG nho di: DUNG LAI, nhieu '
        'khong den tu beta — kiem paired baseline sai / prob-logit khong nhat quan / dung '
        'nham fitted beta / lech hang du doan / clipping-chuan hoa khac giua hai duong code.  '
        '==> Neu |dAP| CO nho dung ti le nhung van tren nguong: day la san luong tu hoa cua '
        'AP, khong phai loi; nguong scale can dieu chinh va ghi thanh amendment.')

# moi B-arm phai dung CUNG beta_init (khong yeu cau cung logits)
assert len({round(v['beta8'], 15) for v in INIT_DIAG.values()}) == 1, 'beta_init khong dong nhat'
print()
print(f'  beta_init dong nhat qua moi B-arm: {1/(1+math.exp(-B_INIT)):.6e}  OK')
print(f'  MOI arm dat near-identity: rho >= {RHO_INIT_MIN} va max|dAP| <= {DAP_INIT_MAX:g}')
print('  LUU Y: nhieu init KHONG triet tieu trong B1-B0 hay B1_CTX-B1 (baseline va u khac nhau),')
print('  nen no duoc chan bang nguong tuyet doi tren TUNG arm, khong dua vao trieu tieu.')
json.dump({k: {kk: vv for kk, vv in v.items()} for k, v in INIT_DIAG.items()},
          open(f'{SC_DIR}/metrics/init_identity_diag.json', 'w'), indent=1)

_B = pd.concat([BETA[(s, a)].assign(seed=s, arm=a) for s in HAVE for a in ('B0', 'B1', 'B1_CTX')])
_B.to_csv(f'{SC_DIR}/metrics/beta_stability.csv', index=False)
print()
print('=== beta_c hoc duoc (arm B1, trung binh qua fold/seed) ===')
_bb = _B[_B.arm == 'B1'][[f'beta_{l}' for l in LABELS]].mean()
print(pd.DataFrame(dict(label=LABELS, tier=TIER, beta_tb=_bb.to_numpy())
                   ).sort_values('beta_tb', ascending=False).to_string(
                   index=False, float_format=lambda v: f'{v:.4f}'))

  seed 42 B0: 20s | lambda_beta [0.01, 0.01, 0.01, 0.01, 0.01]
  seed 42 B1: 21s | lambda_beta [0.01, 0.01, 0.01, 0.01, 0.01]
  seed 42 B1_CTX: 21s | lambda_beta [0.1, 0.1, 0.1, 0.1, 0.1]
  seed 42 B1_SHUF: xong 5 hoan vi

=== PHAN BO |u| (giai thich do lon cua nhieu init) ===
  B0           baseline A0 | p50  0.75  p90  1.59  p99   2.41  p999    3.28  max     5.98  -> max|dz| = beta*max|u| = 2.01e-03
  B1           baseline A1 | p50  0.75  p90  1.59  p99   2.41  p999    3.28  max     5.98  -> max|dz| = beta*max|u| = 2.01e-03
  B1_CTX       baseline A1 | p50  0.80  p90  1.49  p99   2.33  p999    3.88  max     6.95  -> max|dz| = beta*max|u| = 2.33e-03
  B1_SHUF101   baseline A1 | p50  0.75  p90  1.59  p99   2.39  p999    3.31  max     5.04  -> max|dz| = beta*max|u| = 1.69e-03

=== |u| tung nhan (arm B1 = Q_FULL vs B1_CTX = Q_CTX) ===
                            p99        p999          mx       
arm                          B1 B1_CTX   B1 B1_CTX   B1 B1_CTX
label                        

## C7 · Kết quả OOF-VAL · factorial · interaction · cổng

```
G_view        = A1 - A0        G_EHR_fixed   = B0 - A0
G_EHR_learned = B1 - A1        G_combined    = B1 - A0
Interaction   = (B1 - A1) - (B0 - A0)
```

Đọc interaction theo §1: `< 0` ⇒ EHR đang **bù** cho view-pooling bottleneck; `> 0` ⇒ hai
modality **bổ sung**; `≈ 0` ⇒ cộng tính.

Mọi so sánh dùng **chung một bộ resample** subject-cluster nên là paired.

⚠ Với một image seed, cổng *"cùng chiều ≥ 2/3 seed"* (§10) **không đánh giá được**. Cell in rõ
trạng thái đó thay vì lặng lẽ bỏ qua.

In [10]:
ARMS = ['A0', 'AG', 'A1', 'AV', 'B0', 'B1', 'B1_CTX']
ALL_OOF = {**VIEW_OOF, **EHR_OOF}
for s in HAVE:
    ALL_OOF[(s, 'AS')] = np.mean([ALL_OOF[(s, f'AS{p}')] for p in PERM_SEEDS], 0)
    ALL_OOF[(s, 'B1_SHUF')] = np.mean([ALL_OOF[(s, f'B1_SHUF{p}')] for p in PERM_SEEDS], 0)
ARMS_ALL = ARMS+['AS', 'B1_SHUF']
AP_, BT_, VF_ = {}, {}, {}
for s in HAVE:
    for a in ARMS_ALL:
        AP_[(s, a)] = ap_vec(ALL_OOF[(s, a)], V['y'])
        BT_[(s, a)], VF_[(s, a)] = boot_groups(ALL_OOF[(s, a)], V['y'])
print('=== OOF-VAL theo arm (macro | rare3 | nontgt | tail | worst3) ===')
T = pd.DataFrame([dict(seed=s, arm=a, **gm(AP_[(s, a)])) for s in HAVE for a in ARMS_ALL])
print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
T.to_csv(f'{SC_DIR}/metrics/oof_val_metrics.csv', index=False)

CON = [('G_view', 'A1', 'A0'), ('view_primary', 'A1', 'AG'), ('AG_vs_A0', 'AG', 'A0'),
       ('view_content', 'A1', 'AS'), ('view_availability', 'A1', 'AV'),
       ('G_EHR_fixed', 'B0', 'A0'), ('G_EHR_learned', 'B1', 'A1'),
       ('G_combined', 'B1', 'A0'), ('ehr_content', 'B1', 'B1_SHUF'),
       ('clinical_increment', 'B1', 'B1_CTX'), ('context_gain', 'B1_CTX', 'A1')]
rows = []
for s in HAVE:
    for nm, hi, lo in CON:
        for g in GRP:
            r = dci(BT_[(s, hi)], BT_[(s, lo)], AP_[(s, hi)], AP_[(s, lo)], g)
            rows.append(dict(seed=s, contrast=nm, arm_hi=hi, arm_lo=lo, metric=g, **r))
D = pd.DataFrame(rows); D.to_csv(f'{SC_DIR}/metrics/per_label_deltas.csv', index=False)
print()
print('=== CONTRAST (macro / rare3 / nontgt), paired, cluster=subject ===')
for g in ('macro', 'rare3', 'nontgt'):
    print(f'  --- {g} ---')
    for s in HAVE:
        for nm, hi, lo in CON:
            r = D[(D.seed == s) & (D.contrast == nm) & (D.metric == g)].iloc[0]
            print(f'    s{s} {nm:<20} {r.delta:+.5f}  CI [{r.lo:+.5f}, {r.hi:+.5f}]  '
                  f'{"LOAI 0" if r.excl0 else "chua 0"}')

print()
print('=== INTERACTION (§1, §10.4 — khong co nguong) ===')
I = []
for s in HAVE:
    for g in GRP:
        d = (BT_[(s, 'B1')][g]-BT_[(s, 'A1')][g]) - (BT_[(s, 'B0')][g]-BT_[(s, 'A0')][g])
        pt = ((np.nanmean(AP_[(s, 'B1')][GRP[g]])-np.nanmean(AP_[(s, 'A1')][GRP[g]]))
              - (np.nanmean(AP_[(s, 'B0')][GRP[g]])-np.nanmean(AP_[(s, 'A0')][GRP[g]])))
        lo, hi = np.percentile(d, [2.5, 97.5])
        I.append(dict(seed=s, metric=g, interaction=pt, lo=float(lo), hi=float(hi)))
        if g in ('macro', 'rare3'):
            print(f'  s{s} {g:<7} {pt:+.5f}  CI [{lo:+.5f}, {hi:+.5f}]  -> ' +
                  ('EHR bu cho view-pooling bottleneck' if pt < -0.0005 else
                   'hai modality bo sung' if pt > 0.0005 else 'gan cong tinh'))
pd.DataFrame(I).to_csv(f'{SC_DIR}/metrics/factorial_interaction.csv', index=False)

print()
print('=== COÅNG §10 ===')
def _g(s, nm, g):
    r = D[(D.seed == s) & (D.contrast == nm) & (D.metric == g)].iloc[0]; return r
VERD = {}
for s in HAVE:
    v1 = _g(s, 'view_primary', 'macro'); vn = _g(s, 'view_primary', 'nontgt')
    vc = _g(s, 'view_content', 'macro'); va = _g(s, 'view_availability', 'macro')
    e1m = _g(s, 'G_EHR_learned', 'macro'); e1r = _g(s, 'G_EHR_learned', 'rare3')
    en = _g(s, 'G_EHR_learned', 'nontgt'); ec = _g(s, 'ehr_content', 'macro')
    cb = _g(s, 'G_combined', 'macro')
    okv = bool((v1.delta >= 0.001) and (vn.delta >= -0.0015))
    oke = bool(((e1m.delta >= 0.002) or (e1r.delta >= 0.002))
              and (en.delta >= -0.0015) and (ec.delta > 0))
    okc = bool(cb.delta >= 0.002)
    VERD[s] = dict(view=okv, ehr=oke, combined=okc,
                   view_content=bool(vc.delta > 0), view_is_availability=bool(abs(va.delta) < 0.0005))
    print(f'  seed {s}:')
    print(f'    VIEW  A1-AG macro {v1.delta:+.5f} (>= +0.001) | NT {vn.delta:+.5f} (>= -0.0015) '
          f'-> {"PASS" if okv else "FAIL"}')
    print(f'          A1-AS {vc.delta:+.5f} -> {"co the claim lateral CONTENT" if vc.delta > 0 else "KHONG duoc claim content"}')
    print(f'          A1-AV {va.delta:+.5f} -> ' +
          ('A1 ~ AV: gain la ACQUISITION-POLICY/CONTEXT, khong phai lateral content'
           if abs(va.delta) < 0.0005 else 'A1 khac AV'))
    print(f'    EHR   B1-A1 macro {e1m.delta:+.5f} rare3 {e1r.delta:+.5f} | NT {en.delta:+.5f} | '
          f'B1-B1_SHUF {ec.delta:+.5f} -> {"PASS" if oke else "FAIL"}')
    print(f'    COMB  B1-A0 macro {cb.delta:+.5f} (>= +0.002) -> {"PASS" if okc else "FAIL"}')
print()
if len(HAVE) < 2:
    print(f'  ! CHI CO {len(HAVE)} image seed. Cong "cung chieu >= 2/3 seed" (§10) KHONG')
    print(f'    danh gia duoc. Ket qua tren la OOF-VAL mot seed — KHONG du de claim.')
else:
    for nm in ('view_primary', 'G_EHR_learned'):
        sg = [np.sign(_g(s, nm, 'macro').delta) for s in HAVE]
        print(f'  {nm}: cung chieu {max(sg.count(1), sg.count(-1))}/{len(sg)} seed')
json.dump({str(s): v for s, v in VERD.items()}, open(f'{SC_DIR}/metrics/gates_oof_val.json', 'w'), indent=1)
pd.DataFrame([dict(seed=s, arm=a, **{f'vf_{l}': float(x) for l, x in zip(LABELS, VF_[(s, a)])})
              for s in HAVE for a in ARMS_ALL]).to_csv(
    f'{SC_DIR}/metrics/bootstrap_subject_2000.csv', index=False)
pd.DataFrame([dict(seed=s, arm=a, **gm(AP_[(s, a)]))
              for s in HAVE for a in ('AS', 'B1_SHUF')]).to_csv(
    f'{SC_DIR}/metrics/shuffled_controls.csv', index=False)

=== OOF-VAL theo arm (macro | rare3 | nontgt | tail | worst3) ===
 seed     arm  macro  rare3  nontgt   tail  worst3
   42      A0 0.4365 0.1467  0.5234 0.1191  0.1339
   42      AG 0.4365 0.1468  0.5234 0.1192  0.1339
   42      A1 0.4365 0.1468  0.5234 0.1192  0.1339
   42      AV 0.4355 0.1512  0.5208 0.1224  0.1374
   42      B0 0.4374 0.1467  0.5246 0.1191  0.1339
   42      B1 0.4375 0.1468  0.5246 0.1192  0.1339
   42  B1_CTX 0.4369 0.1468  0.5239 0.1192  0.1339
   42      AS 0.4355 0.1512  0.5208 0.1223  0.1374
   42 B1_SHUF 0.4367 0.1468  0.5237 0.1192  0.1339

=== CONTRAST (macro / rare3 / nontgt), paired, cluster=subject ===
  --- macro ---
    s42 G_view               +0.00003  CI [+0.00001, +0.00006]  LOAI 0
    s42 view_primary         +0.00001  CI [-0.00000, +0.00002]  chua 0
    s42 AG_vs_A0             +0.00003  CI [+0.00000, +0.00005]  LOAI 0
    s42 view_content         +0.00104  CI [-0.00005, +0.00222]  chua 0
    s42 view_availability    +0.00102  CI [-0.00005, +0.

## C8 · **TEST — chạy một lần** *(Stage 2, mặc định TẮT)*

§7.1: TEST chỉ chạy **sau khi** khoá arm, regularization, permutation seed, optimization,
metric, go/no-go. §7.3: chọn `λ` bằng **modal / median log-λ** của 5 outer fold, fit toàn bộ
calibration layer trên **full VAL**, dự đoán TEST **một lần**.

Cell tự chặn nếu chưa đủ điều kiện:

| chặn | vì |
|---|---|
| `RUN_TEST = False` mặc định | TEST không đảo ngược được |
| `< 2` image seed | §10 đòi cùng chiều `≥ 2/3` seed |
| thiếu `image_view_probs_seed*_test.npz` | cần forward TEST (~6 phút/seed GPU) |
| thiếu `ehr_*_probs_test.npz` | cần `C3` chạy với `RUN_TEST = True` |

**Sau khi chạy, §13 có hiệu lực**: không đổi label subset, không hard-code `α_Fracture = 1`,
không tắt EHR riêng cho `Pleural_Other`, không đổi ràng buộc `β`, không đổi `λ`, không chọn arm
theo per-label TEST, không thêm patient-specific gate để cứu null.

In [11]:
RUN_TEST = False       # <<< chi bat sau khi Stage 1 da khoa hoan toan

if not RUN_TEST:
    print('C8 TAT. TEST chua duoc cham.')
    print(f'  hash TEST manifest: {HASH["test_sid"][:16]}...')
    print()
    print('Dieu kien truoc khi bat:')
    print(f'  1. image seed co san: {HAVE}  (can >= 2 theo §10)')
    _need = [f'seed{s}_test' for s in IMAGE_SEEDS
             if not os.path.exists(_npz(s, "test"))]
    print(f'  2. thieu prediction TEST: {_need if _need else "khong thieu"}')
    print(f'  3. thieu EHR TEST: '
          f'{[t for t in ("ctx","full") if not os.path.exists(f"{SC_DIR}/predictions/ehr_{t}_probs_test.npz")]}')
    print('  4. Stage 1 da doc va KHOA (arm, lambda, perm seed, cong) — khong sua gi sau day')
else:
    assert len(HAVE) >= 2, (f'chi co {len(HAVE)} image seed; §10 doi cung chieu >= 2/3. '
                            'Chay seed con lai o nb16 truoc.')
    TS = SPL['test']
    for s in HAVE:
        assert os.path.exists(_npz(s, 'test')), f'thieu {_npz(s,"test")}'
    for t in ('ctx', 'full'):
        assert os.path.exists(f'{SC_DIR}/predictions/ehr_{t}_probs_test.npz'), f'thieu EHR TEST {t}'
    QT = {t: np.load(f'{SC_DIR}/predictions/ehr_{t}_probs_test.npz')['q'] for t in ('ctx', 'full')}
    # ---- lambda: modal/median log-lambda cua 5 outer fold (§7.3) ----
    def _pick(vals):
        v = [tuple(x) if isinstance(x, list) else x for x in vals]
        return max(set(v), key=v.count)
    _sut = np.unique(TS['subject']); _sixt = np.searchsorted(_sut, TS['subject'])
    _rt = np.random.default_rng(20240401)
    BOOT_T = [np.bincount(_rt.integers(0, len(_sut), len(_sut)),
                          minlength=len(_sut))[_sixt] for _ in range(N_BOOT)]
    def boot_t(P):
        pr = [_prep(P[:, j]) for j in range(C)]
        yo = [TS['y'][pr[j][0], j].astype('float64') for j in range(C)]
        o = {g: np.empty(N_BOOT) for g in GRP}
        for b, c in enumerate(BOOT_T):
            a = np.array([ap_grouped(yo[j], c[pr[j][0]].astype('float64'), pr[j][1], pr[j][2])
                          for j in range(C)])
            for g, jj in GRP.items(): o[g][b] = np.nanmean(a[jj])
        return o
    ALLV = np.arange(len(V['sid'])); TOOF, TAP, TBT = {}, {}, {}
    for s in HAVE:
        zv, zt = PFL[(s, 'val')], PFL[(s, 'test')]
        lamV = _pick(list(VIEW_PAR[(s, 'A1')].lam))
        TOOF[(s, 'A0')] = pool_alpha(zt['pF'], zt['pL'], TS['mF'], TS['mL'], np.full(C, W_F0))
        for mode, lam in (('AG', _pick(list(VIEW_PAR[(s, 'AG')].lam))), ('A1', lamV),
                          ('AV', _pick(list(VIEW_PAR[(s, 'AV')].lam)))):
            f, _ = make_view_fitter(zv['pF'], zv['pL'], V['mF'], V['mL'], V['y'], mode)
            par = f(ALLV, lam)                                     # fit tren FULL VAL (§7.3)
            _, pt = make_view_fitter(zt['pF'], zt['pL'], TS['mF'], TS['mL'], TS['y'], mode)
            TOOF[(s, mode)] = pt(par, np.arange(len(TS['sid'])))
        for arm, vmode, qv, qt in (('B0', 'A0', Q[('full', 'val')], QT['full']),
                                   ('B1', 'A1', Q[('full', 'val')], QT['full']),
                                   ('B1_CTX', 'A1', Q[('ctx', 'val')], QT['ctx'])):
            lb = _pick(list(BETA[(s, arm)].lam_beta))
            f, p = make_view_fitter(zv['pF'], zv['pL'], V['mF'], V['mL'], V['y'], vmode)
            par = f(ALLV, lamV if vmode == 'A1' else (0.0, 0.0))
            zi = LG(p(par, ALLV)); ui = _u(qv, ALLV, ALLV)
            be = fit_beta(zi, ui, V['y'], lb)
            _, pt = make_view_fitter(zt['pF'], zt['pL'], TS['mF'], TS['mL'], TS['y'], vmode)
            zt_ = LG(pt(par, np.arange(len(TS['sid']))))
            lq = LG(qv); ut = (LG(qt)-lq.mean(0))/(lq.std(0)+1e-8)
            TOOF[(s, arm)] = 1/(1+np.exp(-(zt_+be[None, :]*ut)))
        for a in ('A0', 'AG', 'A1', 'AV', 'B0', 'B1', 'B1_CTX'):
            TAP[(s, a)] = ap_vec(TOOF[(s, a)], TS['y']); TBT[(s, a)] = boot_t(TOOF[(s, a)])
    TT = pd.DataFrame([dict(seed=s, arm=a, **gm(TAP[(s, a)]))
                       for s in HAVE for a in ('A0', 'AG', 'A1', 'AV', 'B0', 'B1', 'B1_CTX')])
    print('=== TEST (mot lan) ==='); print(TT.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    TT.to_csv(f'{SC_DIR}/metrics/test_metrics.csv', index=False)
    print()
    print('=== CONTRAST tren TEST ===')
    rr = []
    for s in HAVE:
        for nm, hi, lo in (('view_primary', 'A1', 'AG'), ('G_view', 'A1', 'A0'),
                           ('G_EHR_learned', 'B1', 'A1'), ('G_combined', 'B1', 'A0'),
                           ('clinical_increment', 'B1', 'B1_CTX')):
            for g in ('macro', 'rare3', 'nontgt'):
                r = dci(TBT[(s, hi)], TBT[(s, lo)], TAP[(s, hi)], TAP[(s, lo)], g)
                rr.append(dict(seed=s, contrast=nm, metric=g, **r))
                print(f'  s{s} {nm:<20} {g:<7} {r["delta"]:+.5f} '
                      f'CI [{r["lo"]:+.5f}, {r["hi"]:+.5f}] {"LOAI 0" if r["excl0"] else "chua 0"}')
    pd.DataFrame(rr).to_csv(f'{SC_DIR}/metrics/test_contrasts.csv', index=False)
    print()
    print('  §13 CO HIEU LUC TU DAY. Khong duoc sua bat ky lua chon nao sau khi da xem TEST.')

C8 TAT. TEST chua duoc cham.
  hash TEST manifest: 0f1abff04c75a2cb...

Dieu kien truoc khi bat:
  1. image seed co san: [42]  (can >= 2 theo §10)
  2. thieu prediction TEST: ['seed42_test', 'seed1337_test', 'seed2024_test']
  3. thieu EHR TEST: ['ctx', 'full']
  4. Stage 1 da doc va KHOA (arm, lambda, perm seed, cong) — khong sua gi sau day


## C9 · `experiment_manifest.json` (§14)

Gói lại toàn bộ thứ cần để tái lập: hash checkpoint, hash cohort, hash manifest VAL/TEST, phân
hoạch fold, lưới regularization, seed hoán vị, code commit, và mọi cổng.

Colab không có git repo nên `code_commit` sẽ `None` — điền tay sau khi commit notebook.

In [12]:
def sha_file(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while True:
            b = f.read(buf)
            if not b: break
            h.update(b)
    return h.hexdigest()
CODE_COMMIT = None
try:
    import subprocess
    CODE_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                                 text=True, timeout=10).stdout.strip() or None
except Exception:
    pass
MANI = dict(
    notebook='20_view_selection_ehr_factorial.ipynb', plan=PREREG['plan'],
    checkpoints={k: sha_file(v) for k, v in (('backbone_04b', BB_CKPT), ('head_H5', H5_CKPT))
                 if os.path.exists(v)},
    image_seeds_available=HAVE, image_seeds_planned=IMAGE_SEEDS, ref_epoch=REF_EPOCH,
    cohort_hash=HASH, fold_assignment=dict(k_outer=K_OUTER, k_inner=K_INNER, group='subject_id',
                                           seed_outer=20240101, seed_inner_base=20240200),
    reg_grids=dict(lambda_g=LAMBDA_G, lambda_d=LAMBDA_D, lambda_beta=LAMBDA_B),
    perm_seeds=PERM_SEEDS, n_boot=N_BOOT, gates=GATES, forbidden_after_test=FORBIDDEN,
    ehr_engine=ENGINE, ehr_ctx_cols=len(NM_C), ehr_full_cols=X_FULL.shape[1],
    demographics_available=DEMO, code_commit=CODE_COMMIT,
    test_touched=bool(globals().get('RUN_TEST', False)),
    deviations=[
        'plan §14 dat ten 19_; dung 20_ vi 19_fov_ablation.ipynb da ton tai',
        f'demographics: {DEMO if DEMO else "KHONG co trong cohort -> Q_CTX thieu muc nay so voi §2.3"}',
        'alpha thuc te thuoc [0.5, 1) — can tren 1.00 chi tiem can voi map 0.5+0.5*sigmoid',
        'AMENDMENT 2026-07-31 (§5.3 + §8): plan dong thoi yeu cau b_c=-8 (beta=3.354e-4) VA exact identity init logits == image-only. Hai dieu bat kha dong thoi. Exact-identity duoc thay bang near-identity khai bao truoc: Spearman >= 0.9999 va max per-label |dAP| <= 1e-4 (= 5% cua cong thuc dung +0.002), kiem rieng tung B-arm voi paired image baseline cua no (B0<-A0, B1/B1_CTX/B1_SHUF<-A1). KHONG doi model, du lieu, hyperparameter, beta init, endpoint hay success gate.',
        'nhieu init KHONG common-mode giua cac B-arm (baseline va vector u khac nhau) -> chan bang nguong tuyet doi tren tung arm, khong dua vao trieu tieu',
        'them scale diagnostic b=-12 (diagnostic only): xac nhan nhieu ti le theo beta',
        'lambda chon bang macro mAP tren inner-held-out (§7.2 khong chi dinh chi so)',
        'Q_CTX: da CHUYEN t1_status, t4_n_prior_positive_ever, t3_recency_bin va mask cua '
        't1/t3_days_since_last_positive sang Q_FULL — chung mang KET QUA cay (02_ehr_features), '
        'khong phai boi canh cham soc. De o CTX se thoi phong phan "shortcut" o §6.',
        f'image seed co san {HAVE} / can {IMAGE_SEEDS}'])
json.dump(MANI, open(f'{SC_DIR}/experiment_manifest.json', 'w'), indent=1)
print('=== experiment_manifest.json ===')
for k in ('image_seeds_available', 'demographics_available', 'ehr_engine', 'code_commit', 'test_touched'):
    print(f'  {k:<26} {MANI[k]}')
print('  deviations:')
for d in MANI['deviations']: print(f'    - {d}')
print()
print(f'-> {SC_DIR}/experiment_manifest.json')
print()
print('=== FILE DA GHI ===')
for r, _, fs in os.walk(SC_DIR):
    for f in sorted(fs):
        print(f'  {os.path.relpath(os.path.join(r, f), SC_DIR)}')

=== experiment_manifest.json ===
  image_seeds_available      [42]
  demographics_available     []
  ehr_engine                 xgboost
  code_commit                None
  test_touched               False
  deviations:
    - plan §14 dat ten 19_; dung 20_ vi 19_fov_ablation.ipynb da ton tai
    - demographics: KHONG co trong cohort -> Q_CTX thieu muc nay so voi §2.3
    - alpha thuc te thuoc [0.5, 1) — can tren 1.00 chi tiem can voi map 0.5+0.5*sigmoid
    - AMENDMENT 2026-07-31 (§5.3 + §8): plan dong thoi yeu cau b_c=-8 (beta=3.354e-4) VA exact identity init logits == image-only. Hai dieu bat kha dong thoi. Exact-identity duoc thay bang near-identity khai bao truoc: Spearman >= 0.9999 va max per-label |dAP| <= 1e-4 (= 5% cua cong thuc dung +0.002), kiem rieng tung B-arm voi paired image baseline cua no (B0<-A0, B1/B1_CTX/B1_SHUF<-A1). KHONG doi model, du lieu, hyperparameter, beta init, endpoint hay success gate.
    - nhieu init KHONG common-mode giua cac B-arm (baseline va vector u 